# Two-stage cascade for NIH ChestX-ray14 — official split, seed 42
Protocol: `evaluation_protocol.yaml` **v2.0.0** (single seed, no TTA).

**Stage 1:** ConvNeXt-Tiny (`M08_CONVNEXT_TINY_ASL_384_CR`) scores all 14 findings.
**Gate:** if every finding is confidently No or Yes, the CNN answer is kept.
**Stage 2:** otherwise RAD-DINO (`M09_RAD_DINO_518_CR`) decides.

```
X-ray → CNN → calibrated scores → gate ─┬─ all 14 confident → CNN answer
                                        └─ any unsure       → RAD-DINO answer
```

### Sessions (Kaggle)
| Session | Flags | Accelerator | Time |
|---|---|---|---|
| **A** | `RUN_CNN=True` | GPU T4 | ~3 h |
| **B** | `RUN_VIT=True` | GPU T4 | ~7.5 h |
| **T** | `RUN_BENCH=True` | GPU T4 | ~10 min |
| **C** | `RUN_ANALYSIS=True` (others False) | CPU | ~10 min |

* The seed-42 runs already exist. Attach their output folder: sessions A/B skip a model whose logits are found.
* Session C needs only the attached outputs (run folders, `split_NIH_OFFICIAL_PV.csv`, `timing_*.json`), not the image dataset.
* Test data is used only in session C. Calibration, thresholds and the cascade gate are fitted on validation only.
* Single seed: results show test-set uncertainty (bootstrap CIs) but not training-run variability.

In [ ]:
# ============================ CONFIG ============================
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")      # determinism; set before CUDA starts
import json, glob, time, math, random, hashlib, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import rankdata
warnings.filterwarnings("ignore")

# ---- what to run in THIS session ----
RUN_CNN      = False    # Session A (GPU): train ConvNeXt-Tiny, export logits
RUN_VIT      = False    # Session B (GPU): train RAD-DINO, export logits
RUN_BENCH    = False    # Session T (GPU): forward-pass latency of both models
RUN_ANALYSIS = True     # Session C (CPU): calibration, thresholds, cascade, metrics, tables
DEBUG        = False    # small subset + 1+1 epochs; folders get a _debug tag and must never be reported

SEED = 42               # the only training seed of this study

PROTOCOL_VERSION = "2.0.0"
OUT_DIR   = os.environ.get("CXR_OUT_DIR", "/kaggle/working")
INPUT_DIR = os.environ.get("CXR_INPUT_DIR", "/kaggle/input")
DATA_ROOT = "/kaggle/input/datasets/nih-chest-xrays/data"      # NIH images + Data_Entry_2017.csv (sessions A/B)
NUM_WORKERS = 4

DATASET_ID, SPLIT_ID = "NIH_CXR14", "NIH_OFFICIAL_PV"
VAL_FRACTION, SPLIT_SEED = 0.15, 42                              # validation = 15 % of train_val patients
EXPECTED_COUNTS = {"train_val": 86524, "test": 25596, "total": 112120}
EXPECTED_LIST_SHA256 = {                                          # SHA-256 of sorted, LF-joined image ids
    "train_val_list.txt": "61fbe896321c1c1c8b75f3e4f3a08e4fef6486d95ef8a667c31d4d60dca6cb81",
    "test_list.txt":      "38ca5ef7f756092946f57c1a59faca882ed589a1ab1f72590b45dc06c6d5e1cc"}
EXPECTED_SPLIT_SHA256 = "b9f2fbc9aa061dafa76994494b92779651dc3ddd2ed3e7b82c70bbf5cf7071dd"   # split of the seed-42 runs

CLASSES = ["Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass", "Nodule", "Pneumonia",
           "Pneumothorax", "Consolidation", "Edema", "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia"]
C = len(CLASSES)

ASL = dict(gamma_neg=4, gamma_pos=1, clip=0.05)
CNN_CFG = dict(
    kind="cnn", model_id="M08_CONVNEXT_TINY_ASL_384_CR", timm_name="convnext_tiny.fb_in22k_ft_in1k",
    img_size=384, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225),
    batch_size=32, warmup_epochs=1, finetune_epochs=8, patience=2,
    lr_head_warmup=1e-3, lr_head=1e-4, lr_backbone=1e-4,
    weight_decay=0.05, drop_path=0.1, use_ema=False, ema_decay=0.999, grad_clip_norm=1.0, asl=ASL, time_budget_h=8.0)
VIT_CFG = dict(
    kind="raddino", model_id="M09_RAD_DINO_518_CR", hf_name="microsoft/rad-dino",
    img_size=518, mean=(0.5307,) * 3, std=(0.2583,) * 3,
    batch_size=12, warmup_epochs=2, finetune_epochs=4, patience=2, unfreeze_last=8, layer_decay=0.8,
    lr_head_warmup=1e-3, lr_head=2e-4, lr_backbone=2e-5,
    weight_decay=0.05, drop_path=0.1, use_ema=True, ema_decay=0.999, grad_clip_norm=1.0, asl=ASL, time_budget_h=9.0)

# ---- evaluation settings (fixed before looking at test) ----
N_BOOT, BOOTSTRAP_SEED, ALPHA = 1000, 20260922, 0.05              # patient-level bootstrap
ECE_BINS = 15
FLAT_T         = [0.70, 0.80, 0.90, 0.95, 0.98]                  # gate: confident if p <= 1-t or p >= t
MISS_BUDGETS   = [0.01, 0.02, 0.05, 0.10]                        # per-class gate variant
MIN_PREC_YES   = 0.80
GATE_TOLERANCE = 0.01          # cheapest cascade whose val macro-F1 is within this of RAD-DINO alone
MIN_SUPPORT_WARN = 100

# ---- timing benchmark (session T) ----
LATENCY_BATCH_SIZES = [1, 32]
WARMUP_ITERS, TIMED_ITERS, TIMING_REPEATS = 50, 200, 3

if DEBUG:
    for cfg in (CNN_CFG, VIT_CFG):
        cfg.update(img_size=224, warmup_epochs=1, finetune_epochs=1, batch_size=16)
    N_BOOT, WARMUP_ITERS, TIMED_ITERS, TIMING_REPEATS = 50, 5, 20, 1
    DATASET_ID += "_debug"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- helpers ----------------
def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def write_json(path, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)

def save_table(df_, name):
    """CSV, UTF-8, full precision (rounding only when displayed)."""
    df_.to_csv(f"{OUT_DIR}/{name}.csv", index=False, float_format="%.17g", na_rep="")

def fmt(x, d=4):
    return "n/a" if x is None or not np.isfinite(x) else f"{np.round(x, d):.{d}f}"

def _walk(bases=(OUT_DIR, INPUT_DIR)):
    """os.walk over outputs and inputs, skipping the 112k-image folders."""
    for base in dict.fromkeys(bases):
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if not d.startswith("images")]
            yield root, dirs, files

def find_file(fname):
    for root, _, files in _walk():
        if fname in files:
            return os.path.join(root, fname)
    return None

def run_folder_name(model_id):
    return f"{DATASET_ID}__{SPLIT_ID}__{model_id}__s{SEED}"

def find_run_dir(model_id):
    """Seed-42 run folder of a model that contains exported logits. Accepts the older names that carry a
    config hash (…__{model_id}__{hash}__s42__r0). More than one distinct folder is an error."""
    hits = {}
    for root, dirs, files in _walk():
        name = os.path.basename(root)
        if ("logits_test.npz" in files and name.startswith(f"{DATASET_ID}__{SPLIT_ID}__{model_id}__")
                and (name.endswith(f"__s{SEED}") or f"__s{SEED}__" in name) and "probe" not in name):
            hits.setdefault(name, root)                    # OUT_DIR is searched first
    assert len(hits) <= 1, f"several seed-{SEED} folders for {model_id}: {sorted(hits)} - attach only one"
    return next(iter(hits.values()), None)

## 1. Data and official split (sessions A and B)
* **test** = every image in `test_list.txt`; **val** = random 15 % of the *patients* in `train_val_list.txt` (seed 42); **train** = the rest.
* The split file must reproduce the hash of the seed-42 runs (`b9f2…`). If it does not (e.g. a library version changed the split), the notebook stops: a model trained on another split is not comparable.
* T1 (dataset characteristics) is written here because it needs the NIH metadata file.

In [ ]:
NEED_DATA = RUN_CNN or RUN_VIT
if NEED_DATA:
    from sklearn.model_selection import GroupShuffleSplit
    meta_csv = os.path.join(DATA_ROOT, "Data_Entry_2017.csv")
    if not os.path.exists(meta_csv):
        meta_csv = next((os.path.join(r, f) for r, _, fs in os.walk(INPUT_DIR) for f in fs if f == "Data_Entry_2017.csv"), meta_csv)
    assert os.path.exists(meta_csv), "Data_Entry_2017.csv not found - check DATA_ROOT"
    DATA_ROOT = os.path.dirname(meta_csv)
    png = glob.glob(os.path.join(DATA_ROOT, "images_*", "images", "*.png")) or \
          [os.path.join(r, f) for r, _, fs in os.walk(DATA_ROOT) for f in fs if f.endswith(".png")]
    df = pd.read_csv(meta_csv)
    df["path"] = df["Image Index"].map({os.path.basename(p): p for p in png})
    assert len(df) == EXPECTED_COUNTS["total"] and df.path.notna().all(), "images missing on disk"
    labels = df["Finding Labels"].str.split("|")
    for c in CLASSES:
        df[c] = labels.apply(lambda L, c=c: c in L).astype(np.float32)
    df = df.sort_values("Image Index").reset_index(drop=True)      # canonical order

    # ---- official list files (QC-01) ----
    def read_list(fname):
        p = os.path.join(DATA_ROOT, fname)
        p = p if os.path.exists(p) else find_file(fname)
        assert p, f"{fname} not found"
        return open(p).read().split()
    tv_ids, te_ids = read_list("train_val_list.txt"), read_list("test_list.txt")
    list_sha = {"train_val_list.txt": sha256_bytes("\n".join(sorted(tv_ids)).encode()),
                "test_list.txt": sha256_bytes("\n".join(sorted(te_ids)).encode())}
    assert (len(tv_ids), len(te_ids)) == (EXPECTED_COUNTS["train_val"], EXPECTED_COUNTS["test"]), "QC-01: list counts"
    assert list_sha == EXPECTED_LIST_SHA256, "QC-01: list files differ from the official lists"
    assert not set(tv_ids) & set(te_ids) and set(tv_ids) | set(te_ids) == set(df["Image Index"]), "QC-01: overlap/coverage"

    # ---- split (identical code to the one that produced the seed-42 runs) ----
    df["split"] = np.where(df["Image Index"].isin(set(te_ids)), "test", "trainval")
    tv_frame = df[df.split == "trainval"]
    _, va = next(GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SPLIT_SEED)
                 .split(tv_frame, groups=tv_frame["Patient ID"]))
    df.loc[tv_frame.index, "split"] = "train"
    df.loc[tv_frame.index[va], "split"] = "val"
    split_text = (df[["Image Index", "Patient ID", "split"]]
                  .rename(columns={"Image Index": "image_index", "Patient ID": "patient_id", "split": "partition"})
                  .sort_values("image_index").to_csv(index=False, lineterminator="\n"))
    SPLIT_SHA256 = sha256_bytes(split_text.encode("utf-8"))
    assert DEBUG or SPLIT_SHA256 == EXPECTED_SPLIT_SHA256, \
        f"split hash {SPLIT_SHA256[:12]} != {EXPECTED_SPLIT_SHA256[:12]}: library versions changed the validation carve-out"
    with open(f"{OUT_DIR}/split_{SPLIT_ID}.csv", "w", encoding="utf-8", newline="\n") as f:
        f.write(split_text)
    pats = {s: set(df.loc[df.split == s, "Patient ID"]) for s in ("train", "val", "test")}
    assert not (pats["train"] & pats["val"] or pats["train"] & pats["test"] or pats["val"] & pats["test"]), "QC-02: patient leakage"

    df_train, df_val, df_test = (df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
    if DEBUG:
        df_train, df_val, df_test = (d.sample(n, random_state=SPLIT_SEED).reset_index(drop=True)
                                     for d, n in ((df_train, 3000), (df_val, 1000), (df_test, 1000)))

    # ---- T1 dataset characteristics ----
    t1 = []
    for part, fr in (("train", df_train), ("val", df_val), ("test", df_test)):
        Y = fr[CLASSES].values
        base = dict(dataset_id=DATASET_ID, split_id=SPLIT_ID, partition=part, n_images=len(fr),
                    n_patients=fr["Patient ID"].nunique(), label_cardinality_mean=Y.sum(1).mean(),
                    n_all_negative_images=int((Y.sum(1) == 0).sum()),
                    images_per_patient_mean=len(fr) / fr["Patient ID"].nunique(),
                    view_PA_fraction=(fr["View Position"] == "PA").mean(), male_fraction=(fr["Patient Gender"] == "M").mean(),
                    age_median=fr["Patient Age"].median(), age_q25=fr["Patient Age"].quantile(.25),
                    age_q75=fr["Patient Age"].quantile(.75))
        t1.append(dict(base, label="ALL", n_positive=int(Y.sum()), prevalence=np.nan))
        t1 += [dict(base, label=c, n_positive=int(Y[:, i].sum()), prevalence=Y[:, i].mean()) for i, c in enumerate(CLASSES)]
    save_table(pd.DataFrame(t1), "T1_dataset_characteristics")
    print(f"split {SPLIT_ID} ({SPLIT_SHA256[:12]}) | train {len(df_train):,} | val {len(df_val):,} | test {len(df_test):,}")

## 2. Model, training and benchmark code (sessions A, B, T)
Deterministic cuDNN, seeded workers; per-epoch log (`train_log.csv`); early stopping on validation macro AUROC; best checkpoint; logits for val and test exported **without TTA**. No test metric is computed during training.

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

def set_determinism():
    torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False
    torch.use_deterministic_algorithms(True, warn_only=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def worker_init_fn(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s); random.seed(s)

def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)

class CXRDataset(Dataset):
    def __init__(self, paths, y, cfg, train):
        self.paths, self.y, s = paths, y.astype(np.float32), cfg["img_size"]
        self.tfm = (T.Compose([T.RandomResizedCrop(s, scale=(0.85, 1.0), ratio=(0.9, 1.1)), T.RandomHorizontalFlip(0.5),
                               T.RandomApply([T.RandomRotation(7)], p=0.5), T.ColorJitter(brightness=0.1, contrast=0.1),
                               T.ToTensor()]) if train else
                    T.Compose([T.Resize((s, s), interpolation=T.InterpolationMode.BILINEAR), T.ToTensor()]))
        self.norm = T.Normalize(cfg["mean"], cfg["std"])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("L")               # grayscale -> [0, 1] -> 3 channels
        return self.norm(self.tfm(img).repeat(3, 1, 1)), self.y[i]

def loader(frame, cfg, train, batch_size=None):
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(CXRDataset(frame["path"].values, frame[CLASSES].values, cfg, train),
                      batch_size=batch_size or cfg["batch_size"], shuffle=train, num_workers=NUM_WORKERS,
                      pin_memory=USE_AMP, persistent_workers=NUM_WORKERS > 0, drop_last=False,
                      worker_init_fn=worker_init_fn, generator=g)

class AsymmetricLoss(nn.Module):
    """ASL (Ridnik et al., 2021)."""
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__()
        self.gn, self.gp, self.clip, self.eps = gamma_neg, gamma_pos, clip, eps

    def forward(self, logits, y):
        p = torch.sigmoid(logits.float())
        p_neg = (p - self.clip).clamp(min=0) if self.clip > 0 else p
        loss_pos = y * torch.log(p.clamp(min=self.eps)) * (1 - p).pow(self.gp)
        loss_neg = (1 - y) * torch.log((1 - p_neg).clamp(min=self.eps)) * p_neg.pow(self.gn)
        return -(loss_pos + loss_neg).sum(1).mean()

class CNNClassifier(nn.Module):
    def __init__(self, cfg, pretrained=True):
        super().__init__()
        import timm
        self.backbone = timm.create_model(cfg["timm_name"], pretrained=pretrained, num_classes=0, drop_path_rate=cfg["drop_path"])
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(self.backbone.num_features, C))

    def forward(self, x):
        return self.head(self.backbone(x))

    def backbone_param_groups(self, cfg):
        return [{"params": list(self.backbone.parameters()), "lr": cfg["lr_backbone"]}]

class RadDinoClassifier(nn.Module):
    """RAD-DINO ViT-B/14 + dual pooling (CLS ⊕ mean patch = 1536-d)."""
    def __init__(self, cfg, pretrained=True):
        super().__init__()
        from transformers import AutoModel, AutoConfig
        self.backbone = (AutoModel.from_pretrained(cfg["hf_name"], drop_path_rate=cfg["drop_path"]) if pretrained else
                         AutoModel.from_config(AutoConfig.from_pretrained(cfg["hf_name"], drop_path_rate=cfg["drop_path"])))
        self.n_reg = getattr(self.backbone.config, "num_register_tokens", 0)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(2 * self.backbone.config.hidden_size, C))

    def forward(self, x):
        h = self.backbone(pixel_values=x).last_hidden_state
        return self.head(torch.cat([h[:, 0], h[:, 1 + self.n_reg:].mean(1)], dim=1))

    def backbone_param_groups(self, cfg):
        layers, k = self.backbone.encoder.layer, cfg["unfreeze_last"]
        n = len(layers)
        groups = [{"params": list(self.backbone.layernorm.parameters()), "lr": cfg["lr_backbone"]}]
        groups += [{"params": list(layers[i].parameters()), "lr": cfg["lr_backbone"] * cfg["layer_decay"] ** (n - 1 - i)}
                   for i in range(n - k, n)]
        return groups

def build_model(cfg, pretrained=True):
    seed_everything(SEED)                                          # seeded head initialisation
    return (CNNClassifier(cfg, pretrained) if cfg["kind"] == "cnn" else RadDinoClassifier(cfg, pretrained)).to(DEVICE)

@torch.no_grad()
def predict_logits(model, dl):
    model.eval()
    out = []
    for x, _ in dl:
        with autocast():
            out.append(model(x.to(DEVICE, non_blocking=True)).float().cpu())
    return torch.cat(out).numpy().astype(np.float32)

def configure_phase(model, cfg, phase):
    """warmup = head only; finetune = head + (part of) the backbone."""
    for p in model.parameters():
        p.requires_grad = False
    groups = [{"params": list(model.head.parameters()), "lr": cfg["lr_head_warmup"] if phase == "warmup" else cfg["lr_head"]}]
    if phase == "finetune":
        groups += model.backbone_param_groups(cfg)
    for g in groups:
        for p in g["params"]:
            p.requires_grad = True
    return torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])

def cosine_schedule(opt, total_steps, warmup_frac=0.03):
    w = max(1, int(total_steps * warmup_frac))
    f = lambda s: (s + 1) / w if s < w else 0.5 * (1 + math.cos(math.pi * (s - w) / max(1, total_steps - w)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def val_macro_auroc(y, s):
    from sklearn.metrics import roc_auc_score
    return float(np.mean([roc_auc_score(y[:, c], s[:, c]) for c in range(C) if 0 < y[:, c].sum() < len(y)]))

def train_model(cfg, exp_dir, model):
    loss_fn = AsymmetricLoss(**cfg["asl"])
    train_dl, val_dl = loader(df_train, cfg, True), loader(df_val, cfg, False, cfg["batch_size"] * 2)
    yv = df_val[CLASSES].values
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    plan = [("warmup", i) for i in range(cfg["warmup_epochs"])] + [("finetune", i) for i in range(cfg["finetune_epochs"])]
    best, best_epoch, bad, ema, phase_now, t0, ep_times, truncated = -1.0, 0, 0, None, None, time.time(), [], False
    for ep, (phase, _) in enumerate(plan, 1):
        if ep_times and (time.time() - t0 + max(ep_times)) / 3600 > cfg["time_budget_h"]:
            print("Stopping: next epoch would exceed the time budget -> truncated=true"); truncated = True
            break
        if phase != phase_now:
            phase_now, bad = phase, 0
            opt = configure_phase(model, cfg, phase)
            n_ep = cfg["warmup_epochs"] if phase == "warmup" else cfg["finetune_epochs"]
            sched = cosine_schedule(opt, n_ep * len(train_dl))
            if phase == "finetune" and cfg["use_ema"]:
                from timm.utils import ModelEmaV2
                ema = ModelEmaV2(model, decay=cfg["ema_decay"])
        if USE_AMP:
            torch.cuda.reset_peak_memory_stats()
        model.train()
        te, run_loss = time.time(), 0.0
        for i, (x, y) in enumerate(train_dl):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with autocast():
                logits = model(x)
            loss = loss_fn(logits, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg["grad_clip_norm"])
            scaler.step(opt); scaler.update(); sched.step()
            if ema is not None:
                ema.update(model)
            run_loss += loss.item()
            if i % 500 == 0:
                print(f"  epoch {ep} step {i}/{len(train_dl)} loss {loss.item():.4f}", flush=True)
        eval_model = ema.module if ema is not None else model                 # EMA weights are evaluated (M09)
        zv = predict_logits(eval_model, val_dl)
        auc = val_macro_auroc(yv, zv)
        ep_times.append(time.time() - te)
        row = dict(epoch=ep, phase=phase, train_loss=run_loss / len(train_dl),
                   val_loss=float(loss_fn(torch.as_tensor(zv), torch.as_tensor(yv))), val_macro_auroc=auc,
                   epoch_seconds=ep_times[-1], peak_gpu_mem_mb=torch.cuda.max_memory_allocated() / 2 ** 20 if USE_AMP else np.nan)
        log = f"{exp_dir}/train_log.csv"
        pd.DataFrame([row]).to_csv(log, mode="a", header=not os.path.exists(log), index=False)
        print(f"Epoch {ep} [{phase}] loss {row['train_loss']:.4f} | val macro AUROC {auc:.4f} | {ep_times[-1] / 60:.1f} min", flush=True)
        if auc > best:                                                        # strict '>' keeps the earlier epoch on ties
            best, best_epoch, bad = auc, ep, 0
            torch.save(eval_model.state_dict(), f"{exp_dir}/checkpoint_best.pt")
        elif phase == "finetune":
            bad += 1
            if bad >= cfg["patience"]:
                print("Early stopping."); break
    model.load_state_dict(torch.load(f"{exp_dir}/checkpoint_best.pt", map_location=DEVICE))
    return model, dict(best_val_macro_auroc=best, best_epoch=best_epoch, epochs_trained=len(ep_times),
                       truncated=truncated, train_wall_clock_h=(time.time() - t0) / 3600)

def run_stage(cfg):
    """Train one model with SEED and export val/test logits. Skips if the seed-42 logits already exist."""
    existing = find_run_dir(cfg["model_id"])
    if existing:
        print(f"{cfg['model_id']}: logits already at {existing} - skipping."); return
    exp_dir = f"{OUT_DIR}/{run_folder_name(cfg['model_id'])}"
    os.makedirs(exp_dir, exist_ok=True)
    set_determinism()
    model = build_model(cfg)
    model, info = train_model(cfg, exp_dir, model)
    for part, frame in (("val", df_val), ("test", df_test)):
        z = predict_logits(model, loader(frame, cfg, False, cfg["batch_size"] * 2))
        np.savez(f"{exp_dir}/logits_{part}.npz", image_index=frame["Image Index"].to_numpy(dtype=str),
                 patient_id=frame["Patient ID"].to_numpy(dtype=np.int64), logits=z,
                 labels=frame[CLASSES].to_numpy(dtype=np.uint8), label_order=json.dumps(CLASSES),
                 split_id=SPLIT_ID, split_sha256=SPLIT_SHA256, model_id=cfg["model_id"], seed=SEED, eval_set=part)
    import timm, transformers
    write_json(f"{exp_dir}/config.json", dict(
        model_id=cfg["model_id"], seed=SEED, split_id=SPLIT_ID, split_sha256=SPLIT_SHA256, protocol_version=PROTOCOL_VERSION,
        model_cfg=cfg, train_info=info, checkpoint_sha256=sha256_file(f"{exp_dir}/checkpoint_best.pt"),
        software=dict(torch=torch.__version__, timm=timm.__version__, transformers=transformers.__version__),
        gpu=torch.cuda.get_device_name(0) if USE_AMP else "cpu"))
    print(f"{cfg['model_id']}: done ({info['train_wall_clock_h']:.2f} h, best epoch {info['best_epoch']})")

def benchmark(cfg):
    """Forward-pass latency per image (fp16, no TTA). Weights are random: latency does not depend on their values."""
    assert USE_AMP, "benchmark needs a GPU"
    model = build_model(cfg, pretrained=False).eval()
    res = dict(model_id=cfg["model_id"], gpu=torch.cuda.get_device_name(0), torch=torch.__version__, amp_dtype="fp16",
               scope="model forward pass on a preloaded tensor, no TTA", weights="random (same architecture)",
               warmup_iters=WARMUP_ITERS, timed_iters=TIMED_ITERS, repeats=TIMING_REPEATS,
               total_params=sum(p.numel() for p in model.parameters()), source="session T benchmark")
    for bs in LATENCY_BATCH_SIZES:
        x = torch.randn(bs, 3, cfg["img_size"], cfg["img_size"], device=DEVICE)
        meds = []
        for _ in range(TIMING_REPEATS):
            with torch.inference_mode(), autocast():
                for _ in range(WARMUP_ITERS):
                    model(x)
                torch.cuda.synchronize()
                ms = []
                for _ in range(TIMED_ITERS):
                    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
                    s.record(); model(x); e.record(); torch.cuda.synchronize()
                    ms.append(s.elapsed_time(e) / bs)
            meds.append(float(np.median(ms)))
        res[f"bs{bs}_median_ms_per_image"], res[f"bs{bs}_repeat_medians"] = float(np.median(meds)), meds
    write_json(f"{OUT_DIR}/timing_{cfg['model_id']}.json", res)
    print(cfg["model_id"], {k: round(v, 2) for k, v in res.items() if k.endswith("median_ms_per_image")})
    del model; torch.cuda.empty_cache()

## 3. Run the GPU sessions

In [ ]:
if RUN_CNN:
    run_stage(CNN_CFG)
if RUN_VIT:
    run_stage(VIT_CFG)
if RUN_BENCH:
    for cfg in (CNN_CFG, VIT_CFG):
        benchmark(cfg)

## 4. Evaluation module (session C)
One implementation for every system. Calibration = per-class Platt on validation logits; thresholds = per-class F1 maximum on calibrated validation probabilities (ties → smallest threshold; no positive F1 → 0.5, flagged). Both are applied unchanged to test.

In [ ]:
# ===================== metrics =====================
HIGHER, LOWER = "higher_is_better", "lower_is_better"
DIRECTION = dict(auroc=HIGHER, auprc=HIGHER, precision=HIGHER, recall=HIGHER, specificity=HIGHER, f1=HIGHER, ece=LOWER,
                 stage2_fraction=LOWER, compute_ratio_vs_raddino=LOWER, labels_missed_by_stage1=LOWER)
THRESHOLD_FREE = {"auroc", "auprc", "ece"}

def auroc_cols(y, s):
    """Mann-Whitney AUROC per class (= sklearn roc_auc_score). No positives or no negatives -> NaN."""
    y = y.astype(bool)
    P = y.sum(0).astype(float); N = len(y) - P
    r = rankdata(s, axis=0)
    with np.errstate(invalid="ignore", divide="ignore"):
        a = ((r * y).sum(0) - P * (P + 1) / 2) / (P * N)
    a[(P == 0) | (N == 0)] = np.nan
    return a

def ap_cols(y, s):
    """Average precision, step-wise (= sklearn average_precision_score)."""
    out = np.full(y.shape[1], np.nan)
    for c in range(y.shape[1]):
        yc = y[:, c].astype(bool); P = yc.sum()
        if P == 0 or P == len(yc):
            continue
        o = np.argsort(-s[:, c], kind="mergesort")
        ss, yy = s[o, c], yc[o]
        last = np.r_[ss[1:] != ss[:-1], True]
        tp = np.cumsum(yy)[last]; n_pred = np.arange(1, len(yy) + 1)[last]
        out[c] = np.sum(np.diff(np.r_[0.0, tp / P]) * (tp / n_pred))
    return out

def ece_cols(y, p, bins=ECE_BINS):
    out = np.zeros(y.shape[1])
    for c in range(y.shape[1]):
        idx = np.minimum((p[:, c] * bins).astype(int), bins - 1)
        out[c] = sum(abs(p[idx == k, c].mean() - y[idx == k, c].mean()) * (idx == k).mean() for k in range(bins) if (idx == k).any())
    return out

def metric_values(y, score, pred, with_ece=False):
    """{(metric, class): value}; class = label name, MACRO (nanmean over classes) or MICRO."""
    y, pred = y.astype(bool), pred.astype(bool)
    P = y.sum(0); N = len(y) - P
    tp = (pred & y).sum(0); fp = (pred & ~y).sum(0); fn = (~pred & y).sum(0); tn = (~pred & ~y).sum(0)
    pc = dict(auroc=auroc_cols(y, score), auprc=ap_cols(y, score),
              precision=np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), 0.0),           # no positive prediction -> 0
              recall=np.where(P > 0, tp / np.maximum(P, 1), np.nan),
              specificity=np.where(N > 0, tn / np.maximum(N, 1), np.nan),
              f1=np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), np.nan))
    if with_ece:
        pc["ece"] = ece_cols(y, score)
    v = {}
    for m, arr in pc.items():
        arr = arr.astype(float)
        v.update({(m, CLASSES[c]): float(arr[c]) for c in range(C)})
        v[(m, "MACRO")] = float(np.nanmean(arr)) if np.isfinite(arr).any() else np.nan
    TP, FP, FN = tp.sum(), fp.sum(), fn.sum()
    v[("precision", "MICRO")] = float(TP / (TP + FP)) if TP + FP else 0.0
    v[("recall", "MICRO")] = float(TP / (TP + FN)) if TP + FN else np.nan
    v[("f1", "MICRO")] = float(2 * TP / (2 * TP + FP + FN)) if 2 * TP + FP + FN else np.nan
    return v

# ---------------- calibration and thresholds (validation only) ----------------
def fit_platt(z, y):
    """p = sigmoid(a_c * logit + b_c), per class, binary NLL, L-BFGS."""
    a, b = np.ones(z.shape[1]), np.zeros(z.shape[1])
    for c in range(z.shape[1]):
        zc, yc = z[:, c].astype(np.float64), y[:, c].astype(np.float64)
        def f(w):
            u = w[0] * zc + w[1]
            g = expit(u) - yc
            return np.mean(np.logaddexp(0, u) - yc * u), np.array([np.mean(g * zc), np.mean(g)])
        a[c], b[c] = minimize(f, [1.0, 0.0], jac=True, method="L-BFGS-B", bounds=[(1e-3, 100), (-50, 50)],
                              options=dict(maxiter=1000)).x
    return a, b

def best_f1_threshold(y, s, tp0=0, fp0=0, fn0=0):
    """Threshold on s maximising F1, given decisions already fixed elsewhere (tp0, fp0, fn0).
    Candidates = unique scores; ties -> smallest threshold. Returns (threshold, f1)."""
    y = y.astype(bool); P = int(y.sum())
    f1_none = 2 * tp0 / max(2 * tp0 + fp0 + fn0 + P, 1)           # option: predict nothing here
    if len(s) == 0:
        return np.inf, f1_none
    order = np.argsort(-s, kind="mergesort")
    s_sorted, y_sorted = s[order], y[order]
    last = np.r_[s_sorted[1:] != s_sorted[:-1], True]
    ctp = np.cumsum(y_sorted)[last]; cfp = np.cumsum(~y_sorted)[last]
    tp, fp, fn = tp0 + ctp, fp0 + cfp, fn0 + P - ctp
    f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    k = int(np.flatnonzero(f1 == f1.max())[-1])
    return (np.inf, f1_none) if f1_none > f1[k] else (float(s_sorted[last][k]), float(f1[k]))

def f1_thresholds(y, s):
    thr, degen = np.zeros(C), np.zeros(C, bool)
    for c in range(C):
        t, f = best_f1_threshold(y[:, c], s[:, c])
        if f == 0 or not np.isfinite(t):
            t, degen[c] = 0.5, True
        thr[c] = t
    return thr, degen

# ---------------- cascade ----------------
def cascade_split(p_cnn, lo, hi, mode):
    """CNN is confident about a finding if p <= lo (No) or p >= hi (Yes). An image is routed if any finding is unsure.
    mode='image': Stage 2 decides all 14 findings of a routed image; 'finding': only the unsure findings."""
    conf = (p_cnn <= lo) | (p_cnn >= hi)
    routed = ~conf.all(axis=1)
    use_s2 = ~conf if mode == "finding" else np.repeat(routed[:, None], p_cnn.shape[1], axis=1)
    return use_s2, routed

def tune_cascade_thresholds(y, p_cnn, p_s2, lo, hi, mode):
    """Stage-2 thresholds maximising overall validation F1 per class, given the CNN's fixed decisions."""
    use_s2, _ = cascade_split(p_cnn, lo, hi, mode)
    cnn_pred = p_cnn >= hi
    thr = np.zeros(y.shape[1])
    for c in range(y.shape[1]):
        m, d = use_s2[:, c], ~use_s2[:, c]
        thr[c] = best_f1_threshold(y[m, c], p_s2[m, c], int((cnn_pred[d, c] & y[d, c]).sum()),
                                   int((cnn_pred[d, c] & ~y[d, c]).sum()), int((~cnn_pred[d, c] & y[d, c]).sum()))[0]
    return thr

def cascade_predict(p_cnn, p_s2, lo, hi, thr, mode):
    use_s2, routed = cascade_split(p_cnn, lo, hi, mode)
    return np.where(use_s2, p_s2 >= thr, p_cnn >= hi), np.where(use_s2, p_s2, p_cnn), use_s2, routed

def perclass_bounds(y, p, max_miss, min_prec, min_count=20):
    """Per-class gate: 'No' below the max_miss quantile of positives; 'Yes' where precision >= min_prec."""
    lo, hi = np.zeros(y.shape[1]), np.full(y.shape[1], np.inf)
    for c in range(y.shape[1]):
        lo[c] = np.quantile(p[y[:, c], c], max_miss)
        order = np.argsort(-p[:, c], kind="mergesort")
        s, yy = p[order, c], y[order, c]
        n = np.arange(1, len(s) + 1)
        last = np.r_[s[1:] != s[:-1], True]
        ok = np.where(last & (np.cumsum(yy) / n >= min_prec) & (n >= min_count))[0]
        if len(ok):
            hi[c] = s[ok[-1]]
        lo[c] = min(lo[c], hi[c] - 1e-9)
    return lo, hi

## 5. Load the seed-42 runs and check them
A run is accepted only if its logits carry the expected split hash and label order, and both models share the exact same validation/test images and labels.

In [ ]:
QC = {}
def qc(id_, ok, detail, severity="error"):
    QC[id_] = dict(passed=bool(ok), severity=severity, detail=str(detail))

if RUN_ANALYSIS:
    RUNS = {}
    for name, cfg in (("cnn", CNN_CFG), ("vit", VIT_CFG)):
        d = find_run_dir(cfg["model_id"])
        assert d, f"no seed-{SEED} logits for {cfg['model_id']} - attach the session A/B output"
        z = {p: np.load(f"{d}/logits_{p}.npz") for p in ("val", "test")}
        RUNS[name] = dict(dir=d, model_id=cfg["model_id"], z={p: z[p]["logits"].astype(np.float64) for p in z}, npz=z)
        print(f"{cfg['model_id']}: {d}")
    cz, vz = RUNS["cnn"]["npz"], RUNS["vit"]["npz"]

    split_path = find_file(f"split_{SPLIT_ID}.csv")
    assert split_path, "split file not found - attach the session A/B output"
    SPLIT_SHA256 = sha256_file(split_path)
    split_df = pd.read_csv(split_path)
    part_n = split_df.partition.value_counts().to_dict()
    qc("QC-01 official split", DEBUG or (part_n.get("test") == EXPECTED_COUNTS["test"]
                                        and part_n.get("train", 0) + part_n.get("val", 0) == EXPECTED_COUNTS["train_val"]),
       f"images per partition {part_n}")
    pp = split_df.groupby("patient_id").partition.nunique()
    qc("QC-02 patient leakage", (pp == 1).all(), f"{int((pp > 1).sum())} patients in more than one partition")
    run_split = {str(RUNS[n]["npz"][p]["split_sha256"]) for n in RUNS for p in ("val", "test")}
    qc("QC-03 split hash", (DEBUG or SPLIT_SHA256 == EXPECTED_SPLIT_SHA256) and run_split == {SPLIT_SHA256},
       f"split file {SPLIT_SHA256[:12]}, logits {sorted(s[:12] for s in run_split)}, expected {EXPECTED_SPLIT_SHA256[:12]}")
    same = all(np.array_equal(cz[p]["image_index"], vz[p]["image_index"]) and np.array_equal(cz[p]["labels"], vz[p]["labels"])
               for p in ("val", "test"))
    test_ids = set(split_df.loc[split_df.partition == "test", "image_index"])
    qc("QC-04 identical eval sets", same and (DEBUG or set(cz["test"]["image_index"]) == test_ids),
       f"val {len(cz['val']['image_index'])}, test {len(cz['test']['image_index'])} images; both models identical: {same}")
    qc("QC-05 label order", all(json.loads(str(RUNS[n]["npz"][p]["label_order"])) == CLASSES for n in RUNS for p in ("val", "test")),
       "label order of exported logits")
    failed = [k for k, v in QC.items() if not v["passed"]]
    assert not failed, f"runs rejected: {failed} -> {[QC[k]['detail'] for k in failed]}"

    yv, yt = cz["val"]["labels"].astype(bool), cz["test"]["labels"].astype(bool)
    pid_test = cz["test"]["patient_id"]

    t1 = find_file("T1_dataset_characteristics.csv")                 # written by session A/B; carried into this output
    if t1 and os.path.abspath(os.path.dirname(t1)) != os.path.abspath(OUT_DIR):
        shutil.copy(t1, OUT_DIR)

    # ---- forward-pass latency (session T output; batch 32 is the compute reference) ----
    def find_timing(model_id):
        """A real benchmark (session T) wins over a provisional copy; otherwise OUT_DIR first."""
        fn = f"timing_{model_id}.json"
        hits = [os.path.join(r, fn) for r, _, fs in _walk() if fn in fs]
        hits.sort(key=lambda q: "PROVISIONAL" in json.load(open(q)).get("source", ""))   # stable sort
        return hits[0] if hits else None
    TIMING = {}
    for cfg in (CNN_CFG, VIT_CFG):
        p = find_timing(cfg["model_id"])
        TIMING[cfg["model_id"]] = json.load(open(p)) if p else {}
        if not p:
            print(f"⚠ timing_{cfg['model_id']}.json not found: compute columns will be empty (run session T)")
        elif "PROVISIONAL" in TIMING[cfg["model_id"]].get("source", ""):
            print(f"⚠ {cfg['model_id']}: provisional timing - {TIMING[cfg['model_id']]['source']}")
    ms = lambda mid, bs=32: TIMING[mid].get(f"bs{bs}_median_ms_per_image", np.nan)
    MS_CNN, MS_VIT = ms(CNN_CFG["model_id"]), ms(VIT_CFG["model_id"])
    ratio = lambda f, bs=32: (ms(CNN_CFG["model_id"], bs) + f * ms(VIT_CFG["model_id"], bs)) / ms(VIT_CFG["model_id"], bs)
    print(f"split {SPLIT_ID} ({SPLIT_SHA256[:12]}) | val {len(yv):,} | test {len(yt):,} images | "
          f"latency bs32: CNN {MS_CNN:.2f} ms, RAD-DINO {MS_VIT:.2f} ms per image")

## 6. Calibration, single models, ensemble, cascade sweep and gate selection (validation only)
* Systems: **CNN alone**, **RAD-DINO alone**, **Average CNN+RAD-DINO** (mean of calibrated probabilities), and the **cascade**.
* The cascade sweep (T4) tries flat gates (t = 0.70 … 0.98) and per-class gates, routing per image or per finding, with RAD-DINO or the average as stage 2.
* **Selection rule:** the gate that sends the **fewest validation images** to stage 2 while its validation macro-F1 is within `GATE_TOLERANCE` of RAD-DINO alone.
* `compute_ratio_vs_raddino` = (CNN time + stage2_fraction × RAD-DINO time) / RAD-DINO time, batch-32 forward pass. Note that routing per finding still runs RAD-DINO on the whole image, so its cost equals routing per image.

In [ ]:
if RUN_ANALYSIS:
    CAL, PROB = {}, {}
    for name in ("cnn", "vit"):
        a, b = fit_platt(RUNS[name]["z"]["val"], yv)                                   # validation only
        CAL[name] = dict(model_id=RUNS[name]["model_id"], method="platt_per_class", fit_on="val", a=a.tolist(), b=b.tolist())
        PROB[name] = {p: expit(a * RUNS[name]["z"][p] + b) for p in ("val", "test")}
    cv, ct, vv, vt = PROB["cnn"]["val"], PROB["cnn"]["test"], PROB["vit"]["val"], PROB["vit"]["test"]
    ev, et = (cv + vv) / 2, (ct + vt) / 2

    SYSTEMS = {}
    for name, sv, st, f2 in (("CNN alone", cv, ct, 0.0), ("RAD-DINO alone", vv, vt, 1.0), ("Average CNN+RAD-DINO", ev, et, 1.0)):
        thr, degen = f1_thresholds(yv, sv)
        cost = MS_CNN / MS_VIT if name == "CNN alone" else (1.0 if name == "RAD-DINO alone" else ratio(1.0))
        g = dict(stage2_fraction=f2, compute_ratio_vs_raddino=cost, labels_missed_by_stage1=np.nan)
        SYSTEMS[name] = dict(val=(sv, sv >= thr), test=(st, st >= thr), thresholds=thr.tolist(),
                             threshold_degenerate=degen.tolist(), glob={"val": g, "test": g})

    # ---- cascade sweep ----
    settings = ([("flat", f"t={t:.2f}", np.full(C, 1 - t), np.full(C, t)) for t in FLAT_T] +
                [("per-class", f"miss<={m:.0%}", *perclass_bounds(yv, cv, m, MIN_PREC_YES)) for m in MISS_BUDGETS])
    sweep, cand = [], []
    for s2_name, s2v, s2t in (("RAD-DINO", vv, vt), ("average", ev, et)):
        for mode in ("image", "finding"):
            for kind, label, lo, hi in settings:
                if s2_name == "average" and (mode != "image" or kind != "flat"):
                    continue
                thr = tune_cascade_thresholds(yv, cv, s2v, lo, hi, mode)
                pv_, sv_, use_v, rv = cascade_predict(cv, s2v, lo, hi, thr, mode)
                pt_, st_, use_t, rt = cascade_predict(ct, s2t, lo, hi, thr, mode)
                mv, mt = metric_values(yv, sv_, pv_), metric_values(yt, st_, pt_)
                sweep.append(dict(
                    stage2_model=s2_name, routing_mode=mode, gate_kind=kind, gate=label,
                    val_stage2_fraction=rv.mean(), test_stage2_fraction=rt.mean(),
                    val_compute_ratio=ratio(rv.mean()), test_compute_ratio=ratio(rt.mean()),
                    val_macro_f1=mv[("f1", "MACRO")], test_macro_f1=mt[("f1", "MACRO")],
                    val_macro_auroc=mv[("auroc", "MACRO")], test_macro_auroc=mt[("auroc", "MACRO")],
                    test_macro_recall=mt[("recall", "MACRO")],
                    val_labels_missed_by_stage1=int((~use_v & ~pv_ & yv).sum()),
                    test_labels_missed_by_stage1=int((~use_t & ~pt_ & yt).sum())))
                cand.append(dict(lo=lo, hi=hi, thr=thr, val=(sv_, pv_), test=(st_, pt_), use_t=use_t))
    T4 = pd.DataFrame(sweep)

    # ---- gate selection on VALIDATION only ----
    ref_f1 = metric_values(yv, *SYSTEMS["RAD-DINO alone"]["val"])[("f1", "MACRO")]
    ok = T4[T4.val_macro_f1 >= ref_f1 - GATE_TOLERANCE]
    pick = (ok.sort_values(["val_stage2_fraction", "val_macro_f1"], ascending=[True, False]).index[0] if len(ok)
            else T4.val_macro_f1.idxmax())
    T4["selected_on_val"] = T4.index == pick
    sel, ch = T4.loc[pick], cand[pick]
    SYSTEMS["Cascade (selected)"] = dict(
        val=ch["val"], test=ch["test"], thresholds=ch["thr"].tolist(), threshold_degenerate=(~np.isfinite(ch["thr"])).tolist(),
        gate=dict(stage2_model=sel.stage2_model, routing_mode=sel.routing_mode, gate_kind=sel.gate_kind, gate=sel.gate,
                  lo=ch["lo"].tolist(), hi=ch["hi"].tolist()), use_t=ch["use_t"],
        glob={es: dict(stage2_fraction=sel[f"{es}_stage2_fraction"], compute_ratio_vs_raddino=sel[f"{es}_compute_ratio"],
                       labels_missed_by_stage1=sel[f"{es}_labels_missed_by_stage1"]) for es in ("val", "test")})
    save_table(T4, "T4_cascade_sweep")
    write_json(f"{OUT_DIR}/calibration.json", CAL)
    write_json(f"{OUT_DIR}/thresholds.json", {k: dict(strategy="f1_max_per_class", thresholds=v["thresholds"],
                                                      threshold_degenerate=v["threshold_degenerate"], gate=v.get("gate"))
                                             for k, v in SYSTEMS.items()})
    print(f"Selected on validation: {sel.stage2_model}, {sel.routing_mode}, {sel.gate} | stage-2 fraction "
          f"val {sel.val_stage2_fraction:.3f} -> test {sel.test_stage2_fraction:.3f} | compute ratio "
          f"val {sel.val_compute_ratio:.3f} -> test {sel.test_compute_ratio:.3f}")
    display(T4.round(4))

## 7. Test metrics with patient-level bootstrap CIs and paired comparisons
* 95 % percentile CIs from `N_BOOT` resamples of **patients** (same resamples for every system). Calibration, thresholds and the gate stay fixed.
* Paired comparisons: difference A − B on the same resamples, two-sided bootstrap p-value (floor 1/(B+1)), **Holm** correction over the 6 system pairs of one metric. Significant = CI excludes 0 **and** Holm p < α.

In [ ]:
SYSTEM_ORDER = ["CNN alone", "RAD-DINO alone", "Average CNN+RAD-DINO", "Cascade (selected)"]
COMPARE = [("auroc", "MACRO"), ("auprc", "MACRO"), ("f1", "MACRO"), ("f1", "MICRO"), ("precision", "MACRO"), ("recall", "MACRO")]

def holm(p):
    p = np.asarray(p, float); m = len(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(np.argsort(p)):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

if RUN_ANALYSIS:
    t0 = time.time()
    groups = list(pd.Series(np.arange(len(pid_test))).groupby(pid_test).indices.values())
    keys = list(metric_values(yt, *SYSTEMS[SYSTEM_ORDER[0]]["test"]).keys())
    kpos = {k: i for i, k in enumerate(keys)}
    boot = np.full((N_BOOT, len(SYSTEM_ORDER), len(keys)), np.nan)
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    for b in range(N_BOOT):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        for j, name in enumerate(SYSTEM_ORDER):
            s, p = SYSTEMS[name]["test"]
            v = metric_values(yt[idx], s[idx], p[idx])
            boot[b, j] = [v[k] for k in keys]
    print(f"{N_BOOT} patient-level resamples in {(time.time() - t0) / 60:.1f} min")

    # ---- T2 results (long format; CIs on test only) ----
    rows = []
    for j, name in enumerate(SYSTEM_ORDER):
        S = SYSTEMS[name]
        for es, y in (("val", yv), ("test", yt)):
            for (m, cls), val in metric_values(y, *S[es], with_ece=True).items():
                lo_ = hi_ = np.nan
                if es == "test" and (m, cls) in kpos:
                    col = boot[:, j, kpos[(m, cls)]]; col = col[np.isfinite(col)]
                    if len(col):
                        lo_, hi_ = np.percentile(col, [2.5, 97.5])
                rows.append(dict(system=name, eval_set=es, metric=m, class_name=cls, value=val, ci_low=lo_, ci_high=hi_,
                                 direction=DIRECTION[m], threshold_strategy="none" if m in THRESHOLD_FREE else "f1_max_per_class"))
            rows += [dict(system=name, eval_set=es, metric=m, class_name="GLOBAL", value=val, ci_low=np.nan, ci_high=np.nan,
                          direction=DIRECTION[m], threshold_strategy="n/a") for m, val in S["glob"][es].items()]
    T2 = pd.DataFrame(rows).assign(dataset_id=DATASET_ID, split_id=SPLIT_ID, split_sha256=SPLIT_SHA256, seed=SEED,
                                   calibration="platt_per_class", n_bootstrap=N_BOOT, protocol_version=PROTOCOL_VERSION)
    save_table(T2, "T2_results")

    # ---- T3 paired comparisons ----
    comps = []
    for m, cls in COMPARE:
        i, fam = kpos[(m, cls)], []
        for a in range(len(SYSTEM_ORDER)):
            for b_ in range(a + 1, len(SYSTEM_ORDER)):
                A_, B_ = SYSTEM_ORDER[b_], SYSTEM_ORDER[a]                      # A vs baseline B
                d = boot[:, b_, i] - boot[:, a, i]; d = d[np.isfinite(d)]
                pa = T2.query("eval_set == 'test' and metric == @m and class_name == @cls").set_index("system").value
                delta = pa[A_] - pa[B_]
                fam.append(dict(metric=m, class_name=cls, system_a=A_, system_b=B_, delta=delta,
                                ci_low=np.percentile(d, 2.5), ci_high=np.percentile(d, 97.5),
                                delta_rel_pct=100 * delta / pa[B_] if pa[B_] else np.nan,
                                p_value=max(2 * min((d <= 0).mean(), (d >= 0).mean()), 1 / (N_BOOT + 1))))
        for f, adj in zip(fam, holm([f["p_value"] for f in fam])):
            f["p_holm"] = adj
            f["significant"] = bool((f["ci_low"] > 0 or f["ci_high"] < 0) and adj < ALPHA)
        comps += fam
    T3 = pd.DataFrame(comps)
    save_table(T3, "T3_paired_comparisons")

    # ---- main table (display only) ----
    q = T2[T2.eval_set == "test"]
    show = [("auroc", "MACRO"), ("auprc", "MACRO"), ("f1", "MACRO"), ("f1", "MICRO"), ("precision", "MACRO"),
            ("recall", "MACRO"), ("specificity", "MACRO"), ("ece", "MACRO"),
            ("stage2_fraction", "GLOBAL"), ("compute_ratio_vs_raddino", "GLOBAL"), ("labels_missed_by_stage1", "GLOBAL")]
    main = pd.DataFrame(index=SYSTEM_ORDER)
    for m, cls in show:
        sub = q[(q.metric == m) & (q.class_name == cls)].set_index("system")
        main[f"{m} {cls.lower()} {'↑' if DIRECTION[m] == HIGHER else '↓'}"] = [
            (f"{fmt(sub.value[s])} [{fmt(sub.ci_low[s])}, {fmt(sub.ci_high[s])}]" if np.isfinite(sub.ci_low[s]) else fmt(sub.value[s]))
            for s in SYSTEM_ORDER]
    print(f"TEST | {SPLIT_ID} ({SPLIT_SHA256[:12]}) | seed {SEED} (single seed) | no TTA | Platt per class | "
          f"F1-max thresholds from val | {N_BOOT} patient-level bootstrap resamples")
    display(main)
    display(T3.round(4))

    # ---- T5 per-class results ----
    sup = yt.sum(0)
    T5 = pd.DataFrame({"finding": CLASSES, "support": sup, "prevalence": sup / len(yt)})
    for sname, tag in zip(SYSTEM_ORDER, ("cnn", "raddino", "avg", "cascade")):
        v = q[(q.system == sname) & q.class_name.isin(CLASSES)]
        for m in ("auroc", "auprc", "f1", "recall"):
            T5[f"{m}_{tag}"] = v[v.metric == m].set_index("class_name").value.reindex(CLASSES).values
    S = SYSTEMS["Cascade (selected)"]
    T5["decided_by_cnn_pct"] = 100 * (~S["use_t"]).mean(0)
    T5["labels_missed_by_stage1"] = (~S["use_t"] & ~S["test"][1] & yt).sum(0)
    T5["wide_uncertainty"] = T5.support < MIN_SUPPORT_WARN
    save_table(T5, "T5_per_class")
    display(T5.round(3))

## 8. Compute (T6) and cost/accuracy trade-off
Latency comes from session T (`timing_*.json`). Cascade and ensemble cost is composed from the member latencies: t_CNN + stage2_fraction × t_RAD-DINO (members run one after the other). Break-even = the stage-2 fraction at which the cascade costs as much as RAD-DINO alone.

In [ ]:
if RUN_ANALYSIS:
    t6 = []
    for name, cfg in (("cnn", CNN_CFG), ("vit", VIT_CFG)):
        t, mid = TIMING[cfg["model_id"]], cfg["model_id"]
        lg = pd.read_csv(f"{RUNS[name]['dir']}/train_log.csv")
        t6.append(dict(system=mid, latency_bs1_ms=ms(mid, 1), latency_bs32_ms=ms(mid, 32), throughput_img_s=1000 / ms(mid, 32),
                       total_params=t.get("total_params"), gpu=t.get("gpu"), timing_source=t.get("source", "missing"),
                       train_hours=lg.epoch_seconds.sum() / 3600, epochs_trained=len(lg),
                       best_epoch=int(lg.epoch[lg.val_macro_auroc.idxmax()]), best_val_macro_auroc=lg.val_macro_auroc.max(),
                       peak_gpu_train_mb=lg.peak_gpu_mem_mb.max()))
    for name in ("Average CNN+RAD-DINO", "Cascade (selected)"):
        g = SYSTEMS[name]["glob"]
        t6.append(dict(system=name, val_stage2_fraction=g["val"]["stage2_fraction"], test_stage2_fraction=g["test"]["stage2_fraction"],
                       latency_bs32_ms=MS_CNN + g["test"]["stage2_fraction"] * MS_VIT,
                       latency_bs1_ms=ms(CNN_CFG["model_id"], 1) + g["test"]["stage2_fraction"] * ms(VIT_CFG["model_id"], 1),
                       val_compute_ratio_bs32=ratio(g["val"]["stage2_fraction"]), test_compute_ratio_bs32=ratio(g["test"]["stage2_fraction"]),
                       test_compute_ratio_bs1=ratio(g["test"]["stage2_fraction"], 1),
                       break_even_stage2_fraction_bs32=1 - MS_CNN / MS_VIT,
                       break_even_stage2_fraction_bs1=1 - ms(CNN_CFG["model_id"], 1) / ms(VIT_CFG["model_id"], 1)))
    T6 = pd.DataFrame(t6)
    save_table(T6, "T6_compute")
    display(T6.round(3))

    # ---- trade-off plot (test), selected gate marked; dashed line = its validation routing ----
    base = T2[(T2.eval_set == "test") & (T2.class_name == "MACRO")]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, metric, col, title in ((axes[0], "f1", "test_macro_f1", "Macro F1 ↑"), (axes[1], "auroc", "test_macro_auroc", "Macro AUROC ↑")):
        for (s2, mode, kind), grp in T4.groupby(["stage2_model", "routing_mode", "gate_kind"]):
            ax.plot(100 * grp.test_stage2_fraction, grp[col], "o-" if kind == "flat" else "s--", label=f"{kind}, {mode}, S2={s2}")
        for name, ls in (("RAD-DINO alone", "-"), ("CNN alone", ":"), ("Average CNN+RAD-DINO", "-.")):
            ax.axhline(base[(base.system == name) & (base.metric == metric)].value.item(), ls=ls, c="gray", label=name)
        s = T4[T4.selected_on_val]
        ax.scatter(100 * s.test_stage2_fraction, s[col], s=180, facecolors="none", edgecolors="red", label="selected on val (test position)")
        ax.axvline(100 * s.val_stage2_fraction.item(), ls="--", c="red", alpha=0.5, label="selected gate: val routing")
        ax.set_xlabel("% of test images sent to RAD-DINO"); ax.set_ylabel(title); ax.grid(alpha=0.3)
        ax.set_title(f"{title} vs routing | {SPLIT_ID} | seed {SEED}")
    axes[0].legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/cascade_tradeoff.png", dpi=150)
    plt.show()

## 9. Quality control and required statements

In [ ]:
if RUN_ANALYSIS:
    qc("QC-06 fitted on validation only", True, "calibration, thresholds and gate use val arrays only (by construction)")
    mism = 0
    for (sys_, es, m), d in T2[T2.class_name.isin(CLASSES)].groupby(["system", "eval_set", "metric"]):
        mac = T2[(T2.system == sys_) & (T2.eval_set == es) & (T2.metric == m) & (T2.class_name == "MACRO")].value
        mism += int(len(mac) == 1 and np.isfinite(mac.item()) and abs(np.nanmean(d.value) - mac.item()) > 1e-12)
    qc("QC-07 macro = mean of per-class", mism == 0, f"{mism} mismatches")
    tsrc = {m: TIMING[m].get("source", "missing") for m in TIMING}
    gpus = {TIMING[m].get("gpu") for m in TIMING}
    qc("QC-08 timing", all(TIMING.values()) and len(gpus) == 1 and not any("PROVISIONAL" in s for s in tsrc.values()),
       f"GPUs {gpus}; sources {tsrc}", severity="warning")
    qc("QC-09 single seed", False, f"seed {SEED} only: CIs cover test-set sampling, not training randomness", severity="warning")
    rep = pd.DataFrame([dict(id=k, **v) for k, v in QC.items()])
    save_table(rep, "qc_report")
    pd.set_option("display.max_colwidth", 200)
    display(rep)
    print("\nRequired statements for the report:")
    print(f"* Official split: YES ({SPLIT_ID}; test = test_list.txt; val = 15% of train_val patients, seed {SPLIT_SEED}; "
          f"split_sha256 {SPLIT_SHA256}).")
    print("* Test data used only in this evaluation; calibration, thresholds and the gate were fitted on validation.")
    print(f"* Single training seed ({SEED}), no TTA; legacy CR/WB numbers are not comparable (different split) and are not in these tables.")
    errors = rep[(~rep.passed) & (rep.severity == "error")]
    if len(errors):
        raise RuntimeError(f"QC errors - tables are not protocol-compliant: {errors.id.tolist()}")
    print("QC: no blocking errors.")